In [1]:
%pip install optuna fastparquet --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/1.9 MB ? eta -:--:--


   ━━━━━━━╺━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.4/1.9 MB 10.6 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 1.9/1.9 MB 31.8 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 23.0 MB/s eta 0:00:00


Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import sys
import glob
import shutil

SRC_ROOT = '/kaggle/input/datasets/mmoselli/mouse-dynamics-ic'
DST_ROOT = '/kaggle/working/mouse-dynamics'

os.makedirs(DST_ROOT, exist_ok=True)

shutil.copytree(
    f'{SRC_ROOT}/src',
    f'{DST_ROOT}/src',
    dirs_exist_ok=True
)

for src_db in glob.glob(f'{SRC_ROOT}/best-parameters/**/*.db', recursive=True):
    relative = os.path.relpath(src_db, f'{SRC_ROOT}/best-parameters')
    dst_db   = os.path.join(DST_ROOT, 'best-parameters', relative)
    os.makedirs(os.path.dirname(dst_db), exist_ok=True)
    if not os.path.exists(dst_db):
        shutil.copy2(src_db, dst_db)

print(f"src/     copiado : {os.path.exists(f'{DST_ROOT}/src')}")
print(f"db files copiados: {len(glob.glob(f'{DST_ROOT}/best-parameters/**/*.db', recursive=True))}")

src/     copiado : True
db files copiados: 111


In [3]:
import os

src = '/kaggle/input/datasets/mmoselli/mouse-dynamics-bogazici-split/split'
dst = '/kaggle/working/datasets/split'

os.makedirs('/kaggle/working/datasets', exist_ok=True)

if os.path.islink(dst) or os.path.exists(dst):
    os.remove(dst)

os.symlink(src, dst)

print(f"aponta para: {os.readlink(dst)}")
print(f"conteúdo: {os.listdir(dst)[:5]}")

aponta para: /kaggle/input/datasets/mmoselli/mouse-dynamics-bogazici-split/split
conteúdo: ['user17', 'user1', 'user6', 'user18', 'user13']


In [4]:
os.chdir('/kaggle/working/mouse-dynamics')

if '/kaggle/working/mouse-dynamics' not in sys.path:
    sys.path.insert(0, '/kaggle/working/mouse-dynamics')

from src.classifiers    import EnumClassifiers
from src.dataset_loaders import EnumDatasets
from src.preprocessors  import EnumPreprocessors
from src.splitters      import EnumSplitters
from src.orchestrator   import Orchestrator

print("Imports OK")
print(f"  EnumClassifiers : {[e.value for e in EnumClassifiers]}")
print(f"  EnumDatasets    : {[e.value for e in EnumDatasets]}")
print(f"  EnumPreprocessors: {[e.value for e in EnumPreprocessors]}")
print(f"  EnumSplitters   : {[e.value for e in EnumSplitters]}")

Imports OK
  EnumClassifiers : ['random-forest', 'mlp', 'knn']
  EnumDatasets    : ['balabit', 'minecraft', 'bogazici']
  EnumPreprocessors: ['minecraft', 'khan']
  EnumSplitters   : ['minecraft', 'fifty_fifty', 'half']


In [5]:
import os

for dataset in ['balabit', 'bogazici', 'minecraft']:
    path = f'/kaggle/working/mouse-dynamics/best-parameters/{dataset}'
    os.makedirs(path, exist_ok=True)
    print(f"criado: {path}")

criado: /kaggle/working/mouse-dynamics/best-parameters/balabit
criado: /kaggle/working/mouse-dynamics/best-parameters/bogazici
criado: /kaggle/working/mouse-dynamics/best-parameters/minecraft


In [6]:
from pathlib import Path
import os

p = Path("../datasets/split")
print(f"existe: {p.exists()}")
print(f"é symlink: {p.is_symlink()}")
print(f"aponta para: {os.readlink(p)}")
print(f"conteúdo: {list(p.iterdir())}")

existe: True
é symlink: True
aponta para: /kaggle/input/datasets/mmoselli/mouse-dynamics-bogazici-split/split
conteúdo: [PosixPath('../datasets/split/user17'), PosixPath('../datasets/split/user1'), PosixPath('../datasets/split/user6'), PosixPath('../datasets/split/user18'), PosixPath('../datasets/split/user13'), PosixPath('../datasets/split/user9'), PosixPath('../datasets/split/user5'), PosixPath('../datasets/split/user2'), PosixPath('../datasets/split/user8'), PosixPath('../datasets/split/user3'), PosixPath('../datasets/split/user12'), PosixPath('../datasets/split/user19'), PosixPath('../datasets/split/user14'), PosixPath('../datasets/split/user16'), PosixPath('../datasets/split/user4'), PosixPath('../datasets/split/user15'), PosixPath('../datasets/split/user11'), PosixPath('../datasets/split/user7'), PosixPath('../datasets/split/user10')]


In [ ]:
import gc

from src.classifiers import load_classifier
from src.utils.experiment_logger import ExperimentLogger
from src.dto.extraction_data import ExtractionData
from src.dto.user_data_dto import UserDataDto
from pandas import read_parquet


ALL_CLASSIFIERS  = [EnumClassifiers.RANDOM_FOREST, EnumClassifiers.MLP, EnumClassifiers.KNN]
ALL_WINDOW_SIZES = [10]
#ALL_WINDOW_SIZES = [10, 50, 100, 150, 200]

total = len(ALL_CLASSIFIERS) * len(ALL_WINDOW_SIZES)
count = 0

for window_size in ALL_WINDOW_SIZES:
    split_path = Path("../datasets/split")
    user_dirs = sorted(split_path.glob("user*"))

    for classifier in ALL_CLASSIFIERS:
        with ExperimentLogger(
            classifier_name=classifier.value,
            dataset_name=EnumDatasets.BOGAZICI.value,
            preprocessor_name=EnumPreprocessors.KHAN.value,
            splitter_name=EnumSplitters.HALF.value,
            preprocessor_window_size=window_size,
            is_debug=False,
        ) as experiment_logger:
            loaded_classifier = load_classifier(classifier, False)
            loaded_classifier.set_experiment_logger(experiment_logger)

            for user_dir in user_dirs:
                user_id = user_dir.stem.replace("user", "")
                if user_id != "18":
                    continue
                
                user_dto = UserDataDto(user_id)
                user_dto.training_sessions = {"_merged": read_parquet(user_dir / "training" / str(window_size)  / "_merged.parquet")}
                user_dto.testing_sessions  = {"_merged": read_parquet(user_dir / "testing" / str(window_size)   / "_merged.parquet")}

                extraction_data = ExtractionData()
                extraction_data.add_user(user_dto)

                loaded_classifier.fit(extraction_data)

                del user_dto, extraction_data
                gc.collect()

        del loaded_classifier
        gc.collect()
        
print("=" * 80)
print("TODOS OS EXPERIMENTOS CONCLUÍDOS")
print("=" * 80)

[3/3] window_size=10


[I 2026-05-17 21:51:44,851] A new study created in RDB with name: random-forest_bogazici__user1.db


  0%|          | 0/15 [00:00<?, ?it/s]

[I 2026-05-17 21:57:00,388] Trial 1 finished with value: 0.8124448121467424 and parameters: {'n_estimators': 250, 'max_depth': 5, 'min_samples_split': 4, 'min_samples_leaf': 12, 'max_features': 'log2', 'criterion': 'entropy', 'bootstrap': True, 'class_weight': None, 'max_samples': 0.9086832374604386}. Best is trial 1 with value: 0.8124448121467424.


[I 2026-05-17 21:58:51,661] Trial 0 finished with value: 0.8553701020462116 and parameters: {'n_estimators': 100, 'max_depth': 8, 'min_samples_split': 10, 'min_samples_leaf': 15, 'max_features': 'sqrt', 'criterion': 'entropy', 'bootstrap': False, 'class_weight': 'balanced_subsample'}. Best is trial 0 with value: 0.8553701020462116.


...

[I 2026-05-18 05:37:30,462] Trial 8 finished with value: 0.7682215246745087 and parameters: {'n_estimators': 150, 'max_depth': 10, 'min_samples_split': 15, 'min_samples_leaf': 5, 'max_features': 'log2', 'criterion': 'entropy', 'bootstrap': True, 'class_weight': None, 'max_samples': 0.787342008644401}. Best is trial 0 with value: 0.7994552182841375.


[I 2026-05-18 06:25:00,093] Trial 9 finished with value: 0.7993963217033856 and parameters: {'n_estimators': 400, 'max_depth': 20, 'min_samples_split': 17, 'min_samples_leaf': 20, 'max_features': 'sqrt', 'criterion': 'gini', 'bootstrap': True, 'class_weight': 'balanced_subsample', 'max_samples': 0.5847823420012}. Best is trial 0 with value: 0.7994552182841375.


[I 2026-05-18 06:25:26,520] Trial 5 finished with value: 0.7929528952215475 and parameters: {'n_estimators': 400, 'max_depth': 15, 'min_samples_split': 10, 'min_samples_leaf': 5, 'max_features': 'log2', 'criterion': 'entropy', 'bootstrap': False, 'class_weight': 'balanced_subsample'}. Best is trial 0 with value: 0.7994552182841375.


[I 2026-05-18 06:50:30,470] Trial 11 finished with value: 0.7612652256685779 and parameters: {'n_estimators': 400, 'max_depth': 9, 'min_samples_split': 11, 'min_samples_leaf': 11, 'max_features': 'log2', 'criterion': 'gini', 'bootstrap': True, 'class_weight': 'balanced', 'max_samples': 0.6490742404299079}. Best is trial 0 with value: 0.7994552182841375.


[I 2026-05-18 07:03:30,525] Trial 7 finished with value: 0.7820628980233822 and parameters: {'n_estimators': 150, 'max_depth': 8, 'min_samples_split': 19, 'min_samples_leaf': 19, 'max_features': 0.5, 'criterion': 'gini', 'bootstrap': False, 'class_weight': 'balanced'}. Best is trial 0 with value: 0.7994552182841375.
